In [1]:
!nvidia-smi

Thu Mar 19 11:16:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Min Deep
# ================================
# Minimal DeepLabV3+ Training
# ================================

import os
import torch
import torchvision
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn, optim
import cv2

# -------- CONFIG --------
NUM_CLASSES = 3
BATCH_SIZE = 8
EPOCHS = 5
LR = 1e-4
IMAGE_SIZE = 640
VAL_SPLIT = 0.3

DATA_ROOT = "PATH_TO_DATA"
IMG_DIR = os.path.join(DATA_ROOT, "images")
MASK_DIR = os.path.join(DATA_ROOT, "masks")

# -------- DATASET --------
class DatasetSeg(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        name = self.images[idx]

        img = Image.open(os.path.join(self.img_dir, name)).resize((IMAGE_SIZE, IMAGE_SIZE))
        img = np.array(img).astype(np.float32)

        img = (img - img.min()) / (img.max() - img.min() + 1e-6)
        if img.ndim == 2:
            img = np.stack([img]*3, axis=-1)

        img = torch.from_numpy(img).permute(2, 0, 1)

        mask = cv2.imread(
            os.path.join(self.mask_dir, name.rsplit(".",1)[0] + ".png"),
            cv2.IMREAD_GRAYSCALE
        )
        mask = cv2.resize(mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = torch.from_numpy(mask).long()

        return img, mask

# -------- LOAD DATA --------
dataset = DatasetSeg(IMG_DIR, MASK_DIR)
val_len = int(len(dataset) * VAL_SPLIT)
train_len = len(dataset) - val_len

train_ds, val_ds = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# -------- MODEL --------
device = "cuda" if torch.cuda.is_available() else "cpu"

model = torchvision.models.segmentation.deeplabv3_mobilenet_v3_large(weights="DEFAULT")
model.classifier = torchvision.models.segmentation.deeplabv3.DeepLabHead(960, NUM_CLASSES)
model.aux_classifier = None
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)

# -------- TRAIN --------
print("🚀 Training started")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)["out"]
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # -------- VALIDATION --------
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)

            outputs = model(imgs)["out"]
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == masks).sum().item()
            total += masks.numel()

    acc = 100 * correct / total

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f} | Val Acc: {acc:.2f}%")

print("✅ Training complete")

🚀 Training started
Epoch 1/3 | Loss: 1.0575 | Val Acc: 2.95%
Epoch 2/3 | Loss: 1.0288 | Val Acc: 3.52%
Epoch 3/3 | Loss: 1.0068 | Val Acc: 8.95%
✅ Training complete


In [3]:
# Min Seg
# ================================
# Minimal SegFormer Training
# ================================

import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn, optim
from transformers import SegformerForSemanticSegmentation

# -------- CONFIG --------
NUM_CLASSES = 3
BATCH_SIZE = 8
EPOCHS = 5
LR = 5e-5
IMAGE_SIZE = 640
VAL_SPLIT = 0.3

DATA_ROOT = "PATH_TO_DATA"
IMG_DIR = os.path.join(DATA_ROOT, "images")
MASK_DIR = os.path.join(DATA_ROOT, "masks")

# -------- DATASET --------
class DatasetSeg(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        name = self.images[idx]

        img = Image.open(os.path.join(self.img_dir, name)).resize((IMAGE_SIZE, IMAGE_SIZE))
        img = np.array(img).astype(np.float32)

        img = (img - img.min()) / (img.max() - img.min() + 1e-6)
        if img.ndim == 2:
            img = np.stack([img]*3, axis=-1)

        img = torch.from_numpy(img).permute(2, 0, 1)

        mask = Image.open(
            os.path.join(self.mask_dir, name.rsplit(".",1)[0] + ".png")
        ).convert("L").resize((IMAGE_SIZE, IMAGE_SIZE))

        mask = torch.from_numpy(np.array(mask)).long()

        return img, mask

# -------- LOAD DATA --------
dataset = DatasetSeg(IMG_DIR, MASK_DIR)
val_len = int(len(dataset) * VAL_SPLIT)
train_len = len(dataset) - val_len

train_ds, val_ds = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# -------- MODEL --------
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)

# -------- TRAIN --------
print("🚀 Training started")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        outputs = model(imgs).logits

        masks_resized = torch.nn.functional.interpolate(
            masks.unsqueeze(1).float(),
            size=outputs.shape[2:],
            mode="nearest"
        ).squeeze(1).long()

        loss = criterion(outputs, masks_resized)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # -------- VALIDATION --------
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)

            outputs = model(imgs).logits
            preds = torch.argmax(outputs, dim=1)

            masks_resized = torch.nn.functional.interpolate(
                masks.unsqueeze(1).float(),
                size=preds.shape[1:],
                mode="nearest"
            ).squeeze(1).long()

            correct += (preds == masks_resized).sum().item()
            total += masks_resized.numel()

    acc = 100 * correct / total

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f} | Val Acc: {acc:.2f}%")

print("✅ Training complete")

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b0-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([3]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 256, 1, 1]) in the checkpoint and torch.Size([3, 256, 1, 1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🚀 Training started
Epoch 1/3 | Loss: 1.1590 | Val Acc: 2.55%
Epoch 2/3 | Loss: 1.1383 | Val Acc: 2.56%
Epoch 3/3 | Loss: 1.1365 | Val Acc: 37.23%
✅ Training complete
